In [1]:
import time
import requests
import pandas as pd

BASE_URL = "https://fantasy.premierleague.com/api/"
HEADERS = {"User-Agent": "fpl-mini-league-analytics (personal project)"}
LEAGUE_ID = 548302

def get(endpoint):
    response = requests.get(BASE_URL + endpoint, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()

bootstrap = get("bootstrap-static/")
events = pd.DataFrame(bootstrap["events"])[["id", "finished", "data_checked", "is_current"]]
print(events[events["id"].between(8, 12)])

players = {p["id"]: p["web_name"] for p in bootstrap["elements"]}
league = get(f"leagues-classic/{LEAGUE_ID}/standings/")
manager_ids = [row["entry"] for row in league["standings"]["results"]]
print(len(manager_ids), "managers")

    id  finished  data_checked  is_current
7    8      True          True       False
8    9      True          True       False
9   10      True          True        True
10  11     False         False       False
11  12     False         False       False
15 managers


In [2]:
GW = 10
picks = get(f"entry/{manager_ids[0]}/event/{GW}/picks/")
live = get(f"event/{GW}/live/")
live_points = {e["id"]: e["stats"]["total_points"] for e in live["elements"]}
live_minutes = {e["id"]: e["stats"]["minutes"] for e in live["elements"]}

print("Keys:", list(picks.keys()))
print("Chip:", picks["active_chip"])
print("Auto subs:", picks["automatic_subs"])
print("History:", picks["entry_history"])

rows = pd.DataFrame(picks["picks"])
rows["player"] = rows["element"].map(players)
rows["minutes"] = rows["element"].map(live_minutes)
rows["points"] = rows["element"].map(live_points)
rows["scored"] = rows["points"] * rows["multiplier"]
print(rows.to_string(index=False))
print("Calculated:", rows["scored"].sum(), "| FPL says:", picks["entry_history"]["points"])

Keys: ['active_chip', 'automatic_subs', 'entry_history', 'picks']
Chip: None
Auto subs: []
History: {'event': 10, 'points': 59, 'total_points': 677, 'rank': 4329358, 'rank_sort': 4346913, 'overall_rank': 309978, 'percentile_rank': 40, 'overall_rank_percentage': '3', 'bank': 11, 'value': 1022, 'event_transfers': 1, 'event_transfers_cost': 0, 'points_on_bench': 9}
 element  position  multiplier  is_captain  is_vice_captain  element_type     player  minutes  points  scored
     109         1           1       False            False             1 Verbruggen       90       6       6
      31         2           1       False            False             2      Konsa       90       8       8
       8         3           1       False            False             2  Calafiori       68       6       6
     229         4           1       False            False             2  Tarkowski       90       2       2
     115         5           1       False            False             2  De Cuyper 

In [3]:
finished = events.loc[events["data_checked"], "id"].tolist()
live_by_gw = {}
for gw in finished:
    live_by_gw[gw] = {e["id"]: e["stats"]["total_points"] for e in get(f"event/{gw}/live/")["elements"]}
    time.sleep(1)

results = []
for manager_id in manager_ids:
    for gw in finished:
        try:
            p = get(f"entry/{manager_id}/event/{gw}/picks/")
        except requests.HTTPError:
            continue  # manager hadn't joined yet
        calc = sum(live_by_gw[gw].get(x["element"], 0) * x["multiplier"] for x in p["picks"])
        results.append({
            "gw": gw,
            "chip": p["active_chip"],
            "auto_subs": len(p["automatic_subs"]),
            "captain_mult": next(x["multiplier"] for x in p["picks"] if x["is_captain"]),
            "calc": calc,
            "fpl": p["entry_history"]["points"],
        })
        time.sleep(0.5)

check = pd.DataFrame(results)
check["diff"] = check["calc"] - check["fpl"]
print(len(check), "manager-gameweeks checked")
print("Mismatches:", (check["diff"] != 0).sum())
print(check[check["diff"] != 0].to_string(index=False))
print(check.groupby("auto_subs").agg(rows=("diff", "size"), mismatches=("diff", lambda d: (d != 0).sum())))
print(check["chip"].value_counts(dropna=False))
print(check["captain_mult"].value_counts())

145 manager-gameweeks checked
Mismatches: 0
Empty DataFrame
Columns: [gw, chip, auto_subs, captain_mult, calc, fpl, diff]
Index: []
           rows  mismatches
auto_subs                  
0            86           0
1            53           0
2             6           0
chip
NaN         112
wildcard     13
3xc          10
freehit       6
bboost        4
Name: count, dtype: int64
captain_mult
2    129
3     10
0      6
Name: count, dtype: int64


In [4]:
import duckdb

con = duckdb.connect("../data/warehouse.duckdb", read_only=True)

In [5]:
con.sql("""
    SELECT
        team_name,
        count(*) AS gameweeks,
        CAST(sum(captain_points_lost) AS BIGINT) AS points_lost,
        round(sum(captain_points_lost) / count(*), 1) AS lost_per_gameweek,
        count(*) FILTER (WHERE captain_points_lost = 0) AS best_pick_weeks
    FROM mart_captain_decisions
    GROUP BY team_name
    ORDER BY points_lost DESC
""").df()

,team_name,gameweeks,points_lost,lost_per_gameweek,best_pick_weeks
0,Number One,10,73,7.3,2
1,GibothyFC,10,71,7.1,2
2,Roushy FC,10,71,7.1,1
3,the Busby babes,10,67,6.7,3
4,Last Minute Pick,10,60,6.0,3
5,El 3atawla FC,10,57,5.7,2
6,Deadlock,10,51,5.1,2
7,Youssef Returns,10,51,5.1,2
8,AcunaMorata,10,47,4.7,3
9,OH FERRAN,10,43,4.3,2


In [6]:
con.sql("""
    WITH pick_points AS (
        SELECT p.manager_id, p.gameweek, p.squad_position, p.picked_to_start, pts.points
        FROM stg_manager_picks AS p
        JOIN stg_gameweek_player_points AS pts
            ON pts.gameweek = p.gameweek AND pts.player_id = p.player_id
    )
    SELECT
        manager_id,
        gameweek,
        max(points) FILTER (WHERE squad_position <= 11) AS best_in_final_xi,
        max(points) FILTER (WHERE picked_to_start) AS best_in_picked_xi
    FROM pick_points
    GROUP BY manager_id, gameweek
    HAVING best_in_final_xi <> best_in_picked_xi
    ORDER BY gameweek, manager_id
""").df()

,manager_id,gameweek,best_in_final_xi,best_in_picked_xi
0,2883181,5,14,6
1,8956299,5,8,6
2,8908463,8,9,8


In [7]:
con.sql("""
    SELECT gameweek, chip, captain_name, captain_points, vice_captain_stepped_in,
           best_option_name, best_option_points, captain_points_lost
    FROM mart_captain_decisions
    WHERE team_name = 'H-Awy FC'
    ORDER BY gameweek
""").df()

,gameweek,chip,captain_name,captain_points,vice_captain_stepped_in,best_option_name,best_option_points,captain_points_lost
0,1,bboost,B.Fernandes,2,False,João Pedro,11,9
1,2,NaN,B.Fernandes,23,False,B.Fernandes,23,0
2,3,NaN,Haaland,9,False,Haaland,9,0
3,4,freehit,Palmer,5,False,João Pedro,12,7
4,5,wildcard,Haaland,6,False,Haaland,6,0
5,6,NaN,Haaland,6,False,João Pedro,12,6
6,7,NaN,Haaland,13,False,Haaland,13,0
7,8,NaN,Tavernier,10,True,Tavernier,10,0
8,9,NaN,Haaland,2,False,Konsa,10,8
9,10,NaN,Haaland,2,False,N.Jackson,11,9


In [8]:
con.close()